In [1]:
!pip install -q transformers accelerate lightgbm

In [2]:
# ============================================================
# PHASE 1.2C — ESM2_t12_35M TRUNCATED EMBEDDING BASELINE
# Colab CUDA/T4 optimized
# ============================================================

import os
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)

print("--- Hardware Check ---")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

--- Hardware Check ---
Torch version: 2.11.0+cu128
CUDA available: True
Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [3]:
import os
from pathlib import Path
from google.colab import drive

# ============================================================
# GOOGLE DRIVE PATHS
# ============================================================

# 1. Kết nối cơ bản (Nếu đã mount rồi thì Colab sẽ tự động bỏ qua, không gây lỗi)
try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

# 2. Định nghĩa lại đường dẫn gốc
PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Tự động phát hiện nếu bạn viết hoa chữ 'Model' hoặc viết thường chữ 'model'
actual_model_folder = "model"
if not (PROJECT_DIR / actual_model_folder).exists() and (PROJECT_DIR / "Model").exists():
    actual_model_folder = "Model"

# 3. Gán đường dẫn cấu hình
SPLIT_DIR = PROJECT_DIR / actual_model_folder / "phase1_protein_baseline" / "splits"
BASE_DIR = PROJECT_DIR / actual_model_folder / "phase1_2_esm2_t12_truncated_embedding"

EMBED_DIR = BASE_DIR / "embeddings"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

# 4. Tự động tạo cấu trúc thư mục đầu ra
for folder in [BASE_DIR, EMBED_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Split dir:", SPLIT_DIR)
print("Output dir:", BASE_DIR)

# 5. Kiểm tra an toàn để chắc chắn không bị lỗi AssertionError nữa
if not SPLIT_DIR.exists():
    print("\n❌ THÔNG BÁO: Vẫn không thấy thư mục dữ liệu splits.")
    print("Danh sách các mục thực tế đang có trong thư mục gốc dự án của bạn:")
    if PROJECT_DIR.exists():
        print(os.listdir(PROJECT_DIR))
    else:
        print(f"Không tìm thấy cả thư mục gốc: {PROJECT_DIR}")
else:
    print("\n✅ THÀNH CÔNG: Đường dẫn hoàn toàn hợp lệ! Hãy chạy tiếp các ô lệnh sau.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Split dir: /content/drive/MyDrive/Project_Protein/model/phase1_protein_baseline/splits
Output dir: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding

✅ THÀNH CÔNG: Đường dẫn hoàn toàn hợp lệ! Hãy chạy tiếp các ô lệnh sau.


In [4]:
# ============================================================
# LOAD SAVED SPLITS
# ============================================================

train_path = SPLIT_DIR / "train_protein_v1.csv"
val_path = SPLIT_DIR / "val_protein_v1.csv"
test_path = SPLIT_DIR / "test_protein_v1.csv"

assert train_path.exists(), train_path
assert val_path.exists(), val_path
assert test_path.exists(), test_path

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (1274, 35)
Validation: (273, 35)
Test: (273, 35)

Train labels:
label
0    637
1    637
Name: count, dtype: int64

Validation labels:
label
1    137
0    136
Name: count, dtype: int64

Test labels:
label
0    137
1    136
Name: count, dtype: int64


In [5]:
# ============================================================
# CLEAN SEQUENCES
# Keep only 20 standard amino acids
# ============================================================

STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AAS])
    return seq


for split_df in [train_df, val_df, test_df]:
    split_df["sequence_clean"] = split_df["sequence"].apply(clean_sequence)
    split_df["sequence_clean_length"] = split_df["sequence_clean"].str.len()

print("Train length summary:")
display(train_df["sequence_clean_length"].describe())

print("Validation length summary:")
display(val_df["sequence_clean_length"].describe())

print("Test length summary:")
display(test_df["sequence_clean_length"].describe())

Train length summary:


,sequence_clean_length
count,1274.000000
mean,744.512559
std,643.266085
min,41.000000
25%,354.000000
50%,555.000000
75%,926.000000
max,7388.000000


Validation length summary:


,sequence_clean_length
count,273.000000
mean,869.728938
std,2114.423720
min,56.000000
25%,370.000000
50%,576.000000
75%,977.000000
max,34350.000000


Test length summary:


,sequence_clean_length
count,273.000000
mean,774.432234
std,689.578849
min,51.000000
25%,326.000000
50%,574.000000
75%,1035.000000
max,4861.000000


In [6]:
# ============================================================
# LOAD ESM2_t12_35M MODEL
# ============================================================

ESM_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
MODEL_SHORT_NAME = "ESM2_t12_35M"
REPRESENTATION_NAME = "ESM2_t12_35M_mean_pool_truncated_1022"

print("Loading tokenizer:", ESM_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)

print("Loading model:", ESM_MODEL_NAME)

if device.type == "cuda":
    esm_model = AutoModel.from_pretrained(
        ESM_MODEL_NAME,
        torch_dtype=torch.float16
    )
else:
    esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME)

esm_model.to(device)
esm_model.eval()

print("Model loaded.")
print("Representation:", REPRESENTATION_NAME)
print("Device:", device)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

Loading tokenizer: facebook/esm2_t12_35M_UR50D


config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Representation: ESM2_t12_35M_mean_pool_truncated_1022
Device: cuda
GPU memory allocated: 65.24951171875 MB
GPU memory reserved: 86.0 MB


In [7]:
# ============================================================
# EMBEDDING CONFIG
# ============================================================

MAX_AA_LENGTH = 1022
BATCH_SIZE = 8          # If CUDA OOM, reduce to 4
SAVE_EVERY = 50         # checkpoint every 50 proteins

print("Max amino acid length:", MAX_AA_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Save every:", SAVE_EVERY)

Max amino acid length: 1022
Batch size: 8
Save every: 50


In [8]:
# ============================================================
# BATCH EMBEDDING FUNCTION
# ============================================================

@torch.no_grad()
def embed_sequence_batch_truncated(
    seqs,
    tokenizer,
    model,
    device,
    max_aa_length=1022
):
    """
    Convert a batch of protein sequences into ESM embeddings.

    Policy:
    - Sequences are pre-cleaned.
    - Each sequence is truncated to max_aa_length amino acids.
    - Tokenize with special tokens.
    - Mean-pool amino-acid token embeddings.
    - Exclude special beginning/end tokens from pooling.
    """
    seqs = [str(seq)[:max_aa_length] for seq in seqs]

    encoded = tokenizer(
        seqs,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True,
        truncation=True
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    if device.type == "cuda":
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(**encoded)
    else:
        outputs = model(**encoded)

    hidden = outputs.last_hidden_state        # [batch, token_len, hidden_dim]
    attention_mask = encoded["attention_mask"]

    batch_embeddings = []

    for i in range(hidden.shape[0]):
        mask = attention_mask[i].bool()
        valid_positions = torch.where(mask)[0]

        # Exclude special tokens if possible
        if len(valid_positions) > 2:
            aa_positions = valid_positions[1:-1]
        else:
            aa_positions = valid_positions

        aa_hidden = hidden[i, aa_positions, :]
        emb = aa_hidden.mean(dim=0)

        batch_embeddings.append(emb.detach().float().cpu().numpy())

    batch_embeddings = np.vstack(batch_embeddings)

    del encoded, outputs, hidden, attention_mask

    return batch_embeddings

In [9]:
# ============================================================
# ROBUST SPLIT EMBEDDING EXTRACTION WITH RESUME
# ============================================================

def extract_truncated_embeddings_for_split_gpu(
    split_df,
    split_name,
    tokenizer,
    model,
    device,
    output_dir,
    sequence_col="sequence_clean",
    max_aa_length=1022,
    batch_size=8,
    save_every=50
):
    output_dir = Path(output_dir)

    final_embedding_path = output_dir / f"esm2_t12_trunc_{split_name}_embeddings.npy"
    final_metadata_path = output_dir / f"esm2_t12_trunc_{split_name}_metadata.csv"

    partial_embedding_path = output_dir / f"esm2_t12_trunc_{split_name}_embeddings_partial.npy"
    partial_metadata_path = output_dir / f"esm2_t12_trunc_{split_name}_metadata_partial.csv"

    # If final exists, load and return
    if final_embedding_path.exists() and final_metadata_path.exists():
        print(f"[{split_name}] Final embeddings already exist. Loading...")
        embeddings = np.load(final_embedding_path)
        metadata = pd.read_csv(final_metadata_path)
        return embeddings, metadata

    split_df_reset = split_df.reset_index(drop=True)

    embeddings = []
    metadata_records = []
    start_idx = 0

    # Resume from partial if available
    if partial_embedding_path.exists() and partial_metadata_path.exists():
        print(f"[{split_name}] Partial checkpoint found. Resuming...")

        partial_embeddings = np.load(partial_embedding_path)
        partial_metadata = pd.read_csv(partial_metadata_path)

        embeddings = [partial_embeddings[i] for i in range(partial_embeddings.shape[0])]
        metadata_records = partial_metadata.to_dict("records")

        start_idx = len(metadata_records)

        print(f"[{split_name}] Resuming from index {start_idx}")

    start_time = time.time()

    for batch_start in range(start_idx, len(split_df_reset), batch_size):
        batch_end = min(batch_start + batch_size, len(split_df_reset))
        batch_df = split_df_reset.iloc[batch_start:batch_end]

        seqs = batch_df[sequence_col].tolist()

        try:
            batch_embeddings = embed_sequence_batch_truncated(
                seqs=seqs,
                tokenizer=tokenizer,
                model=model,
                device=device,
                max_aa_length=max_aa_length
            )

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"CUDA OOM at {split_name} batch {batch_start}:{batch_end}.")
                print("Try reducing BATCH_SIZE.")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            raise e

        for local_i, (_, row) in enumerate(batch_df.iterrows()):
            seq = row[sequence_col]
            emb = batch_embeddings[local_i]

            embeddings.append(emb)

            metadata_records.append({
                "row_index": int(batch_start + local_i),
                "gene_id": row.get("gene_id", None),
                "gene_symbol": row.get("gene_symbol", None),
                "uniprot_accession": row.get("uniprot_accession", None),
                "label": int(row["label"]),
                "sequence_clean_length": len(seq),
                "truncated": len(seq) > max_aa_length,
                "used_length": min(len(seq), max_aa_length),
                "max_aa_length": max_aa_length,
                "representation": REPRESENTATION_NAME
            })

        processed = len(metadata_records)

        # Checkpoint
        if processed % save_every == 0 or processed == len(split_df_reset):
            current_embeddings = np.vstack(embeddings)
            current_metadata = pd.DataFrame(metadata_records)

            np.save(partial_embedding_path, current_embeddings)
            current_metadata.to_csv(partial_metadata_path, index=False)

            elapsed = time.time() - start_time

            print(
                f"[{split_name}] {processed}/{len(split_df_reset)} proteins "
                f"| embeddings={current_embeddings.shape} "
                f"| elapsed={elapsed/60:.2f} min"
            )

            if device.type == "cuda":
                torch.cuda.empty_cache()

    final_embeddings = np.vstack(embeddings)
    final_metadata = pd.DataFrame(metadata_records)

    np.save(final_embedding_path, final_embeddings)
    final_metadata.to_csv(final_metadata_path, index=False)

    print(f"[{split_name}] Final embeddings shape:", final_embeddings.shape)
    print(f"[{split_name}] Saved:", final_embedding_path)
    print(f"[{split_name}] Metadata saved:", final_metadata_path)

    return final_embeddings, final_metadata

In [10]:
# ============================================================
# DEBUG RUN — 10 PROTEINS
# ============================================================

debug_df = train_df.head(10).copy()

X_debug_t12, meta_debug_t12 = extract_truncated_embeddings_for_split_gpu(
    split_df=debug_df,
    split_name="debug10",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=5
)

print("Debug embedding shape:", X_debug_t12.shape)
display(meta_debug_t12)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

[debug10] Final embeddings already exist. Loading...
Debug embedding shape: (10, 480)


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,max_aa_length,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,False,101,1022,ESM2_t12_35M_mean_pool_truncated_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,False,463,1022,ESM2_t12_35M_mean_pool_truncated_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,False,319,1022,ESM2_t12_35M_mean_pool_truncated_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,False,267,1022,ESM2_t12_35M_mean_pool_truncated_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,False,309,1022,ESM2_t12_35M_mean_pool_truncated_1022
5,5,ENSG00000213996,TM6SF2,Q9BZW4,1,377,False,377,1022,ESM2_t12_35M_mean_pool_truncated_1022
6,6,ENSG00000164823,OSGIN2,Q9Y236,0,505,False,505,1022,ESM2_t12_35M_mean_pool_truncated_1022
7,7,ENSG00000185716,MOSMO,Q8NHV5,0,167,False,167,1022,ESM2_t12_35M_mean_pool_truncated_1022
8,8,ENSG00000181873,IBA57,Q5T440,0,356,False,356,1022,ESM2_t12_35M_mean_pool_truncated_1022
9,9,ENSG00000164647,STEAP1,Q9UHE8,1,339,False,339,1022,ESM2_t12_35M_mean_pool_truncated_1022


GPU memory allocated: 65.24951171875 MB
GPU memory reserved: 86.0 MB


In [11]:
# ============================================================
# EXTRACT VALIDATION EMBEDDINGS
# ============================================================

X_val_t12, meta_val_t12 = extract_truncated_embeddings_for_split_gpu(
    split_df=val_df,
    split_name="val",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=SAVE_EVERY
)

[val] Final embeddings already exist. Loading...


In [12]:
# ============================================================
# EXTRACT TEST EMBEDDINGS
# ============================================================

X_test_t12, meta_test_t12 = extract_truncated_embeddings_for_split_gpu(
    split_df=test_df,
    split_name="test",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=SAVE_EVERY
)

[test] Final embeddings already exist. Loading...


In [13]:
# ============================================================
# EXTRACT TRAIN EMBEDDINGS
# ============================================================

X_train_t12, meta_train_t12 = extract_truncated_embeddings_for_split_gpu(
    split_df=train_df,
    split_name="train",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=SAVE_EVERY
)

[train] Final embeddings already exist. Loading...


In [14]:
# ============================================================
# LOAD SAVED ESM2_t12 TRUNCATED EMBEDDINGS
# ============================================================

X_train_t12 = np.load(EMBED_DIR / "esm2_t12_trunc_train_embeddings.npy")
X_val_t12 = np.load(EMBED_DIR / "esm2_t12_trunc_val_embeddings.npy")
X_test_t12 = np.load(EMBED_DIR / "esm2_t12_trunc_test_embeddings.npy")

meta_train_t12 = pd.read_csv(EMBED_DIR / "esm2_t12_trunc_train_metadata.csv")
meta_val_t12 = pd.read_csv(EMBED_DIR / "esm2_t12_trunc_val_metadata.csv")
meta_test_t12 = pd.read_csv(EMBED_DIR / "esm2_t12_trunc_test_metadata.csv")

y_train = meta_train_t12["label"].astype(int)
y_val = meta_val_t12["label"].astype(int)
y_test = meta_test_t12["label"].astype(int)

print("X_train_t12:", X_train_t12.shape)
print("X_val_t12:", X_val_t12.shape)
print("X_test_t12:", X_test_t12.shape)

print("\nTrain labels:")
print(y_train.value_counts())

display(meta_train_t12.head())

X_train_t12: (1274, 480)
X_val_t12: (273, 480)
X_test_t12: (273, 480)

Train labels:
label
0    637
1    637
Name: count, dtype: int64


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,max_aa_length,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,False,101,1022,ESM2_t12_35M_mean_pool_truncated_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,False,463,1022,ESM2_t12_35M_mean_pool_truncated_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,False,319,1022,ESM2_t12_35M_mean_pool_truncated_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,False,267,1022,ESM2_t12_35M_mean_pool_truncated_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,False,309,1022,ESM2_t12_35M_mean_pool_truncated_1022


In [15]:
# ============================================================
# TRUNCATION SUMMARY
# ============================================================

truncation_summary = []

for split_name, meta in [
    ("train", meta_train_t12),
    ("validation", meta_val_t12),
    ("test", meta_test_t12)
]:
    truncation_summary.append({
        "split": split_name,
        "n": len(meta),
        "n_truncated": int(meta["truncated"].sum()),
        "pct_truncated": meta["truncated"].mean() * 100,
        "mean_length": meta["sequence_clean_length"].mean(),
        "median_length": meta["sequence_clean_length"].median(),
        "max_length": meta["sequence_clean_length"].max(),
        "mean_used_length": meta["used_length"].mean()
    })

t12_truncation_summary_df = pd.DataFrame(truncation_summary)

display(t12_truncation_summary_df)

t12_truncation_summary_df.to_csv(
    RESULT_DIR / "esm2_t12_truncation_summary_v1.csv",
    index=False
)

,split,n,n_truncated,pct_truncated,mean_length,median_length,max_length,mean_used_length
0,train,1274,263,20.643642,744.512559,555.0,7388,605.529827
1,validation,273,65,23.809524,869.728938,576.0,34350,621.208791
2,test,273,69,25.274725,774.432234,574.0,4861,608.311355


In [16]:
# ============================================================
# FREE GPU MEMORY AFTER EMBEDDING
# ============================================================

del esm_model
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared.")

GPU memory cleared.


In [17]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [18]:
# ============================================================
# CV + SCORING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [19]:
# ============================================================
# DOWNSTREAM MODELS FOR ESM2_t12 EMBEDDINGS
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


t12_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    t12_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    t12_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

print("Models:")
for name in t12_models:
    print("-", name)

Models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM


In [20]:
# ============================================================
# BASELINE CV BEFORE TUNING
# ============================================================

t12_baseline_cv_results = []

for model_name, pipeline in t12_models.items():
    print("=" * 80)
    print("ESM2_t12 baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train_t12,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),

        "overfit_gap_train_minus_cv": (
            cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        )
    }

    t12_baseline_cv_results.append(row)

t12_baseline_cv_df = pd.DataFrame(t12_baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(t12_baseline_cv_df)

t12_baseline_cv_df.to_csv(
    RESULT_DIR / "esm2_t12_baseline_cv_before_tuning_v1.csv",
    index=False
)

ESM2_t12 baseline CV: Logistic Regression
ESM2_t12 baseline CV: SVM RBF
ESM2_t12 baseline CV: Random Forest
ESM2_t12 baseline CV: LightGBM


,representation,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
1,ESM2_t12_35M_mean_pool_truncated_1022,SVM RBF,0.938034,0.002500,0.708995,0.026347,0.642593,0.029399,0.705570,0.021046,0.295533,0.055699,0.229040
2,ESM2_t12_35M_mean_pool_truncated_1022,Random Forest,1.000000,0.000000,0.702315,0.033847,0.644518,0.030680,0.698435,0.031621,0.288969,0.058554,0.297685
3,ESM2_t12_35M_mean_pool_truncated_1022,LightGBM,1.000000,0.000000,0.687339,0.022154,0.637213,0.014130,0.675601,0.023030,0.273525,0.024523,0.312661
0,ESM2_t12_35M_mean_pool_truncated_1022,Logistic Regression,0.954273,0.004192,0.652623,0.031516,0.618561,0.032062,0.634072,0.053777,0.249772,0.058388,0.301650


In [21]:
# ============================================================
# PARAMETER GRIDS — COLAB OPTIMIZED
# ============================================================

t12_param_grids = {
    "Logistic Regression": {
        "model__C": [0.003, 0.01, 0.03, 0.1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.1, 1, 10],
        "model__gamma": [0.001, 0.01, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2],
        "model__bootstrap": [True]
    }
}


if LIGHTGBM_AVAILABLE:
    t12_param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.05],
        "model__num_leaves": [7, 15],
        "model__max_depth": [3, 5],
        "model__min_child_samples": [20, 50],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0.1],
        "model__reg_lambda": [1, 5]
    }
else:
    t12_param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.03, 0.05],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31],
        "model__min_samples_leaf": [20, 50]
    }

In [22]:
# ============================================================
# RUN GRIDSEARCHCV
# ============================================================

t12_grid_search_results = []
t12_best_estimators = {}

for model_name, pipeline in t12_models.items():
    print("=" * 80)
    print("ESM2_t12 GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=t12_param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_t12, y_train)

    t12_best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    t12_grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

t12_grid_results_df = pd.DataFrame(t12_grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(t12_grid_results_df)

t12_grid_results_df.to_csv(
    RESULT_DIR / "esm2_t12_gridsearch_results_v1.csv",
    index=False
)

ESM2_t12 GridSearchCV: Logistic Regression
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV ROC-AUC: 0.7038687108624218
Best train ROC-AUC: 0.7963922293752816
Gap: 0.09252351851285989
Best params: {'model__C': 0.003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
ESM2_t12 GridSearchCV: SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.7089947648645298
Best train ROC-AUC: 0.9380343392181789
Gap: 0.22903957435364908
Best params: {'model__C': 1, 'model__gamma': 'scale'}
ESM2_t12 GridSearchCV: Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best CV ROC-AUC: 0.7084001449252899
Best train ROC-AUC: 0.9909497858236481
Gap: 0.2825496408983582
Best params: {'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 500}
ESM2_t12 GridSearchCV: LightGBM
Fitting 5 folds for each of 128 candidates, totalling 640 fits
Best CV ROC-AUC: 0.703

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
1,ESM2_t12_35M_mean_pool_truncated_1022,SVM RBF,0.708995,0.938034,0.229040,"{'model__C': 1, 'model__gamma': 'scale'}"
2,ESM2_t12_35M_mean_pool_truncated_1022,Random Forest,0.708400,0.990950,0.282550,"{'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 500}"
0,ESM2_t12_35M_mean_pool_truncated_1022,Logistic Regression,0.703869,0.796392,0.092524,"{'model__C': 0.003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
3,ESM2_t12_35M_mean_pool_truncated_1022,LightGBM,0.703174,0.999996,0.296822,"{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__min_child_samples': 50, 'model__n_estimators': 200, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model__reg_lambda': 1, 'model__subsample': 0.8}"


In [28]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

t12_tuned_eval_records = []

for model_name, model in t12_best_estimators.items():
    print("=" * 80)
    print("Evaluating:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train_t12,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val_t12,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    train_metrics["representation"] = REPRESENTATION_NAME
    val_metrics["representation"] = REPRESENTATION_NAME

    t12_tuned_eval_records.extend([train_metrics, val_metrics])

t12_tuned_eval_df = pd.DataFrame(t12_tuned_eval_records)

display(t12_tuned_eval_df)

t12_validation_comparison = t12_tuned_eval_df[
    t12_tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(t12_validation_comparison)

t12_tuned_eval_df.to_csv(
    RESULT_DIR / "esm2_t12_tuned_train_validation_metrics_v1.csv",
    index=False
)

t12_validation_comparison.to_csv(
    RESULT_DIR / "esm2_t12_validation_comparison_v1.csv",
    index=False
)

Evaluating: Logistic Regression
Evaluating: SVM RBF
Evaluating: Random Forest
Evaluating: Soft Voting
Evaluating: Stacking


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,train,0.730769,0.728972,0.734694,0.726845,0.731822,0.790469,0.784145,0.461553,463,174,169,468,ESM2_t12_35M_mean_pool_truncated_1022
1,Logistic Regression,validation,0.688645,0.706349,0.649635,0.727941,0.676806,0.747209,0.738352,0.378696,99,37,48,89,ESM2_t12_35M_mean_pool_truncated_1022
2,SVM RBF,train,0.864207,0.871795,0.854003,0.874411,0.862807,0.932962,0.931789,0.728566,557,80,93,544,ESM2_t12_35M_mean_pool_truncated_1022
3,SVM RBF,validation,0.663004,0.680000,0.620438,0.705882,0.648855,0.729229,0.718595,0.327482,96,40,52,85,ESM2_t12_35M_mean_pool_truncated_1022
4,Random Forest,train,0.956829,0.951863,0.962323,0.951334,0.957065,0.989198,0.990836,0.913713,606,31,24,613,ESM2_t12_35M_mean_pool_truncated_1022
5,Random Forest,validation,0.652015,0.650000,0.664234,0.639706,0.657040,0.727780,0.730994,0.304037,87,49,46,91,ESM2_t12_35M_mean_pool_truncated_1022
6,Soft Voting,train,0.863422,0.868045,0.857143,0.869702,0.862559,0.931175,0.932621,0.726902,554,83,91,546,ESM2_t12_35M_mean_pool_truncated_1022
7,Soft Voting,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.747638,0.738161,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
8,Stacking,train,0.870487,0.874603,0.864992,0.875981,0.869771,0.941336,0.942957,0.741018,558,79,86,551,ESM2_t12_35M_mean_pool_truncated_1022
9,Stacking,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.745975,0.737401,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
7,Soft Voting,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.747638,0.738161,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
1,Logistic Regression,validation,0.688645,0.706349,0.649635,0.727941,0.676806,0.747209,0.738352,0.378696,99,37,48,89,ESM2_t12_35M_mean_pool_truncated_1022
9,Stacking,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.745975,0.737401,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
3,SVM RBF,validation,0.663004,0.680000,0.620438,0.705882,0.648855,0.729229,0.718595,0.327482,96,40,52,85,ESM2_t12_35M_mean_pool_truncated_1022
5,Random Forest,validation,0.652015,0.650000,0.664234,0.639706,0.657040,0.727780,0.730994,0.304037,87,49,46,91,ESM2_t12_35M_mean_pool_truncated_1022


In [24]:
# ============================================================
# SOFT VOTING
# ============================================================

t12_voting_estimators = []

for model_name, estimator in t12_best_estimators.items():
    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    t12_voting_estimators.append((short_name, estimator))

t12_soft_voting = VotingClassifier(
    estimators=t12_voting_estimators,
    voting="soft",
    n_jobs=-1
)

t12_soft_voting.fit(X_train_t12, y_train)

t12_voting_train_metrics = evaluate_binary_classifier(
    t12_soft_voting,
    X_train_t12,
    y_train,
    "Soft Voting",
    "train"
)

t12_voting_val_metrics = evaluate_binary_classifier(
    t12_soft_voting,
    X_val_t12,
    y_val,
    "Soft Voting",
    "validation"
)

t12_voting_train_metrics["representation"] = REPRESENTATION_NAME
t12_voting_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([t12_voting_train_metrics, t12_voting_val_metrics]))

t12_best_estimators["Soft Voting"] = t12_soft_voting

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,train,0.863422,0.868045,0.857143,0.869702,0.862559,0.931175,0.932621,0.726902,554,83,91,546,ESM2_t12_35M_mean_pool_truncated_1022
1,Soft Voting,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.747638,0.738161,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022


In [25]:
# ============================================================
# STACKING
# ============================================================

t12_stacking_estimators = []

for model_name, estimator in t12_best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    t12_stacking_estimators.append((short_name, estimator))

t12_stacking = StackingClassifier(
    estimators=t12_stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

t12_stacking.fit(X_train_t12, y_train)

t12_stacking_train_metrics = evaluate_binary_classifier(
    t12_stacking,
    X_train_t12,
    y_train,
    "Stacking",
    "train"
)

t12_stacking_val_metrics = evaluate_binary_classifier(
    t12_stacking,
    X_val_t12,
    y_val,
    "Stacking",
    "validation"
)

t12_stacking_train_metrics["representation"] = REPRESENTATION_NAME
t12_stacking_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([t12_stacking_train_metrics, t12_stacking_val_metrics]))

t12_best_estimators["Stacking"] = t12_stacking

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Stacking,train,0.870487,0.874603,0.864992,0.875981,0.869771,0.941336,0.942957,0.741018,558,79,86,551,ESM2_t12_35M_mean_pool_truncated_1022
1,Stacking,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.745975,0.737401,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022


In [29]:
# ============================================================
# FINAL VALIDATION COMPARISON
# ============================================================

t12_all_eval_df = pd.concat(
    [
        t12_tuned_eval_df,
        pd.DataFrame([
            t12_voting_train_metrics,
            t12_voting_val_metrics,
            t12_stacking_train_metrics,
            t12_stacking_val_metrics
        ])
    ],
    ignore_index=True
)

t12_all_validation_results = t12_all_eval_df[
    t12_all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(t12_all_validation_results)

t12_all_eval_df.to_csv(
    RESULT_DIR / "esm2_t12_all_train_validation_metrics_v1.csv",
    index=False
)

t12_all_validation_results.to_csv(
    RESULT_DIR / "esm2_t12_all_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
7,Soft Voting,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.747638,0.738161,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
11,Soft Voting,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.747638,0.738161,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
1,Logistic Regression,validation,0.688645,0.706349,0.649635,0.727941,0.676806,0.747209,0.738352,0.378696,99,37,48,89,ESM2_t12_35M_mean_pool_truncated_1022
9,Stacking,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.745975,0.737401,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
13,Stacking,validation,0.670330,0.685039,0.635036,0.705882,0.659091,0.745975,0.737401,0.341745,96,40,50,87,ESM2_t12_35M_mean_pool_truncated_1022
3,SVM RBF,validation,0.663004,0.680000,0.620438,0.705882,0.648855,0.729229,0.718595,0.327482,96,40,52,85,ESM2_t12_35M_mean_pool_truncated_1022
5,Random Forest,validation,0.652015,0.650000,0.664234,0.639706,0.657040,0.727780,0.730994,0.304037,87,49,46,91,ESM2_t12_35M_mean_pool_truncated_1022


In [30]:
# ============================================================
# SELECT FINAL MODEL FROM VALIDATION
# ============================================================

t12_best_validation_row = t12_all_validation_results.iloc[0]
final_t12_model_name = t12_best_validation_row["model"]
final_t12_model = t12_best_estimators[final_t12_model_name]

print("Final selected ESM2_t12 truncated model:", final_t12_model_name)
display(t12_best_validation_row)

pd.DataFrame([t12_best_validation_row]).to_csv(
    RESULT_DIR / "esm2_t12_final_model_validation_summary_v1.csv",
    index=False
)

Final selected ESM2_t12 truncated model: Soft Voting


,7
model,Soft Voting
dataset,validation
accuracy,0.67033
precision,0.685039
recall_sensitivity,0.635036
specificity,0.705882
f1,0.659091
roc_auc,0.747638
pr_auc,0.738161
mcc,0.341745


In [31]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

t12_final_test_metrics = evaluate_binary_classifier(
    model=final_t12_model,
    X=X_test_t12,
    y=y_test,
    model_name=final_t12_model_name,
    dataset_name="test"
)

t12_final_test_metrics["representation"] = REPRESENTATION_NAME

t12_final_test_metrics_df = pd.DataFrame([t12_final_test_metrics])

display(t12_final_test_metrics_df)

t12_final_test_metrics_df.to_csv(
    RESULT_DIR / "esm2_t12_final_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,test,0.549451,0.544218,0.588235,0.510949,0.565371,0.587484,0.594175,0.099478,70,67,56,80,ESM2_t12_35M_mean_pool_truncated_1022


In [32]:
# ============================================================
# DIAGNOSTIC TEST ALL MODELS
# Not used for final model selection.
# ============================================================

t12_diagnostic_test_records = []

for model_name, model in t12_best_estimators.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test_t12,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic"
    )

    metrics["representation"] = REPRESENTATION_NAME
    t12_diagnostic_test_records.append(metrics)

t12_diagnostic_test_df = pd.DataFrame(t12_diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(t12_diagnostic_test_df)

t12_diagnostic_test_df.to_csv(
    RESULT_DIR / "esm2_t12_diagnostic_all_models_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,test_diagnostic,0.578755,0.571429,0.617647,0.540146,0.593640,0.613836,0.589747,0.158261,74,63,52,84,ESM2_t12_35M_mean_pool_truncated_1022
3,Soft Voting,test_diagnostic,0.549451,0.544218,0.588235,0.510949,0.565371,0.587484,0.594175,0.099478,70,67,56,80,ESM2_t12_35M_mean_pool_truncated_1022
4,Stacking,test_diagnostic,0.542125,0.537415,0.580882,0.503650,0.558304,0.581902,0.592372,0.084783,69,68,57,79,ESM2_t12_35M_mean_pool_truncated_1022
1,SVM RBF,test_diagnostic,0.545788,0.543478,0.551471,0.540146,0.547445,0.574603,0.596044,0.091621,74,63,61,75,ESM2_t12_35M_mean_pool_truncated_1022
2,Random Forest,test_diagnostic,0.531136,0.527397,0.566176,0.496350,0.546099,0.564191,0.565384,0.062678,68,69,59,77,ESM2_t12_35M_mean_pool_truncated_1022


In [33]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_pred = final_t12_model.predict(X_test_t12)
y_test_score = get_positive_class_score(final_t12_model, X_test_t12)

t12_test_predictions_df = meta_test_t12.copy()
t12_test_predictions_df["true_label"] = y_test.values
t12_test_predictions_df["pred_label"] = y_test_pred
t12_test_predictions_df["pred_score_t2d_associated"] = y_test_score

display(t12_test_predictions_df.head())

t12_test_predictions_df.to_csv(
    RESULT_DIR / "esm2_t12_final_test_predictions_v1.csv",
    index=False
)

,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,max_aa_length,representation,true_label,pred_label,pred_score_t2d_associated
0,0,ENSG00000177542,SLC25A22,Q9H936,0,323,False,323,1022,ESM2_t12_35M_mean_pool_truncated_1022,0,1,0.753791
1,1,ENSG00000123080,CDKN2C,P42773,1,168,False,168,1022,ESM2_t12_35M_mean_pool_truncated_1022,1,1,0.509100
2,2,ENSG00000185262,UBALD2,Q8IYN6,0,164,False,164,1022,ESM2_t12_35M_mean_pool_truncated_1022,0,1,0.656736
3,3,ENSG00000092148,HECTD1,Q9ULT8,0,2610,True,1022,1022,ESM2_t12_35M_mean_pool_truncated_1022,0,1,0.506453
4,4,ENSG00000139364,TMEM132B,Q14DG7,1,1078,True,1022,1022,ESM2_t12_35M_mean_pool_truncated_1022,1,1,0.508700


In [34]:
# ============================================================
# SAVE MODELS
# ============================================================

for model_name, model in t12_best_estimators.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")

    model_path = MODEL_DIR / f"esm2_t12_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)

final_t12_model_path = MODEL_DIR / "esm2_t12_final_selected_model_v1.pkl"
joblib.dump(final_t12_model, final_t12_model_path)

print("Final ESM2_t12 model saved:", final_t12_model_path)

Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding/models/esm2_t12_logistic_regression_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding/models/esm2_t12_svm_rbf_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding/models/esm2_t12_random_forest_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding/models/esm2_t12_soft_voting_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding/models/esm2_t12_stacking_best_estimator_v1.pkl
Final ESM2_t12 model saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_esm2_t12_truncated_embedding/models/esm2_t12_final_selected_model_v1.pkl


In [35]:
# ============================================================
# MASTER COMPARISON TABLE
# Fill handcrafted / ESM2_t6 numbers from previous phases
# ============================================================

master_rows = [
    {
        "phase": "1.1",
        "representation": "AAC + Physchem",
        "embedding_policy": "handcrafted",
        "final_model": "Random Forest",
        "test_roc_auc": 0.5520,
        "test_pr_auc": 0.5550,
        "test_f1": 0.5390,
        "test_mcc": 0.0480
    },
    {
        "phase": "1.2A",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "truncated_1022",
        "final_model": "Stacking",
        "test_roc_auc": 0.6202,
        "test_pr_auc": 0.6188,
        "test_f1": 0.5926,
        "test_mcc": 0.1941
    },
    {
        "phase": "1.2B",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "final_model": "Soft Voting",
        "test_roc_auc": 0.6277,
        "test_pr_auc": 0.6278,
        "test_f1": 0.5870,
        "test_mcc": 0.1650
    },
    {
        "phase": "1.2C",
        "representation": "ESM2_t12_35M",
        "embedding_policy": "truncated_1022",
        "final_model": final_t12_model_name,
        "test_roc_auc": t12_final_test_metrics["roc_auc"],
        "test_pr_auc": t12_final_test_metrics["pr_auc"],
        "test_f1": t12_final_test_metrics["f1"],
        "test_mcc": t12_final_test_metrics["mcc"]
    }
]

master_comparison_df = pd.DataFrame(master_rows)

display(master_comparison_df)

master_comparison_df.to_csv(
    RESULT_DIR / "master_comparison_with_esm2_t12_v1.csv",
    index=False
)

,phase,representation,embedding_policy,final_model,test_roc_auc,test_pr_auc,test_f1,test_mcc
0,1.1,AAC + Physchem,handcrafted,Random Forest,0.552000,0.555000,0.539000,0.048000
1,1.2A,ESM2_t6_8M,truncated_1022,Stacking,0.620200,0.618800,0.592600,0.194100
2,1.2B,ESM2_t6_8M,sliding_window_1022_stride_1022,Soft Voting,0.627700,0.627800,0.587000,0.165000
3,1.2C,ESM2_t12_35M,truncated_1022,Soft Voting,0.587484,0.594175,0.565371,0.099478
